# Notebook Sidebar AI Agent — Integration Architecture

Companion to `2026-06-09-notebook-sidebar-ai-agent-design.md`. An app-aware AI Agent chat in the notebook sidebar: default-scoped to the "notebook app", re-scoped to a Spur App on switch. Apps integrate by contributing **MCP tools + skill**; the one sidebar agent drives them (e.g. paints the Code Graph Workbench panels) and streams a grounded answer.

**Reuse, not reinvention.** Every concern maps to shipped infrastructure:

| Concern | Reused |
|---|---|
| Panel | `SIDEBAR_PANELS` registry + `useSidebar.activatePanel` |
| Sessions (B) | ACP `new_session` / `load_session` / `list_sessions`; `supports_load_session` |
| Discovery | `Orchestrator::list_sessions_from_disk` + `_from_rpc_with_cwd` (merge by id) |
| Re-scope (C) | `new_session(cwd, mcp_servers)` (MCP binds at creation) |
| UX | `SessionPickerView` |
| Turn engine | `AcpAgentBackend` drain loop |
| Streaming | `chat_turn` + `Channel<ChatEvent>` (the `run_cell` pattern) |
| Permissions | TUI-identical: `permission_tx` + `pending_permission` + `skip_permissions` |
| Painting | `notebook_push_source` cascade |

## 1. System context

The sidebar is **trusted React** (direct `invoke`/store access, no sandbox). Its only agent path is an app-owned `NativeAcpConnection`. Apps contribute tools + skill; the agent paints app panels.

```mermaid
flowchart LR
  User([Developer])
  subgraph SB["Sidebar (trusted React)"]
    CP["AI Agent ChatPanel"]
  end
  subgraph BE["Foundation backend (Tauri / spur-notebook)"]
    SC["SidebarChat session mgr<br/>+ chat_turn command"]
  end
  subgraph EXT["ACP / apps"]
    AC["NativeAcpConnection"]
    AG(["Agent subprocess"])
    APP["Spur App: mcp tools + skill"]
    PAN["App AFM panels"]
  end
  User -->|prompt| CP
  CP -->|invoke chat_turn| SC
  SC -->|new/load_session, prompt| AC
  AC -->|spawn + prompt| AG
  AG -->|call tools| APP
  APP -->|notebook_push_source| PAN
  AG -->|SessionNotification stream| AC
  AC -->|ChatEvent| SC
  SC -->|stream| CP
  CP -->|answer + inline permission| User
```

## 2. Integration architecture

The centerpiece: how the sidebar panel, the foundation frontend (registry + notebook store), the Tauri backend, and the ACP/app layer connect.

```mermaid
flowchart TB
  subgraph SB["Sidebar (trusted React)"]
    CP["ChatPanel.tsx<br/>messages + stream + inline permission + session picker"]
    ST["stores/chat.ts"]
  end
  subgraph FE["Foundation frontend"]
    REG["SIDEBAR_PANELS registry<br/>useSidebar.activatePanel('agent')"]
    NS["notebook store<br/>viewState.path / viewMode"]
  end
  subgraph BE["Tauri backend / spur-notebook"]
    TC["chat_turn / chat_* commands<br/>Channel&lt;ChatEvent&gt;"]
    SC["SidebarChat session mgr<br/>(AcpAgentBackend pattern)"]
    CTX["app-context loader<br/>spur-app.json -> cwd / mcp / skill"]
    MCP["Notebook MCP server<br/>(+ proxied app plugin tools)"]
  end
  subgraph EXT["ACP / apps"]
    ACP["NativeAcpConnection"]
    AG(["Agent subprocess"])
    APP["app plugin (wb_*) + skill"]
    PANELS["app AFM panels"]
  end

  CP --> ST
  REG --> CP
  NS -- "path change" --> ST
  CP -- "invoke chat_turn" --> TC
  TC --> SC
  SC -- "scope" --> CTX
  CTX -- "reads" --> APP
  SC -- "new_session / load_session / prompt" --> ACP
  ACP -- "spawn + prompt" --> AG
  AG -- "call tools" --> MCP
  MCP -- "proxy" --> APP
  APP -- "notebook_push_source" --> MCP
  MCP -- "cascade" --> PANELS
  AG -- "SessionNotification" --> ACP
  ACP -- "stream" --> SC
  SC -- "ChatEvent" --> TC
  TC -- "onEvent" --> CP
```

**Facts:** the sidebar agent calls app `wb_*` + foundation tools through the notebook MCP socket (plugin tools proxied additively); painting tools call `notebook_push_source(port, ipc_bytes)` which queues `ReactiveEngine::push_source`. The chat stream reuses the `run_cell` `Channel` pattern.

## 3. Session lifecycle (B + C-refresh, all native)

```mermaid
stateDiagram-v2
  [*] --> NotebookApp: open notebook (default scope)
  NotebookApp --> Scoping: open / switch to a Spur App
  Scoping --> AppSession: new_session(cwd, app mcp + skill)  [C-refresh]
  Scoping --> AppSession: load_session(id)  [resume, if supports_load_session]
  AppSession --> AppSession: prompt turn (stream)
  AppSession --> NotebookApp: switch back to plain notebook
  AppSession --> OtherApp: switch to another app (cancel in-flight, re-scope)
  OtherApp --> AppSession: switch back (load_session replay)
  AppSession --> [*]: window close (Drop killpg)
```

- **B:** sessions are `session_id`-namespaced on one connection, scoped by `cwd`; listed via the Orchestrator's disk+rpc merge.
- **C-refresh:** changing tools = a fresh `new_session` (MCP binds at creation; never in-place mutation).
- **Resume:** `load_session` replays history through the notification broadcast — subscribe BEFORE calling it to capture replay. Gated by `supports_load_session`.

## 4. One chat turn, end to end

```mermaid
sequenceDiagram
  actor U as User
  participant CP as ChatPanel (React)
  participant TC as chat_turn (Tauri)
  participant SC as SidebarChat mgr
  participant ACP as NativeAcpConnection
  participant AG as Agent subprocess
  participant MCP as Notebook MCP (+app plugin)
  participant W as App panels

  U->>CP: type prompt
  CP->>TC: invoke(chat_turn, {sessionRef, prompt, onEvent})
  TC->>SC: ensure app session (load_session | new_session)
  SC->>ACP: prompt(PromptRequest)
  ACP->>AG: PromptRequest
  AG->>MCP: call app wb_* / foundation tools
  MCP-->>W: notebook_push_source -> cascade (panels paint)
  AG-->>ACP: SessionNotification (chunks, tool calls, permission)
  ACP-->>SC: stream items
  SC-->>CP: ChatEvent (MessageChunk / ToolCall / PermissionRequest / Usage)
  CP-->>U: stream answer; inline permission grant/deny
  Note over CP,AG: chat_cancel -> conn.cancel(session_id)
```

Panels paint mid-turn as the agent's tool calls hit `notebook_push_source`, before the prose answer finishes.

## 5. Backend types

```mermaid
classDiagram
  class SidebarChat {
    -conn: AgentConnection
    -sessions: Map~AppKey, SessionId~
    +turn(sessionRef, prompt, channel)
    +list_sessions(cwd)
    +switch(appKey)
    +cancel()
  }
  class AgentConnection {
    <<trait>>
    +new_session(cwd, mcp_servers)
    +load_session(req) Stream~SessionNotification~
    +list_sessions()
    +prompt(req) Stream~SessionNotification~
    +cancel(session_id)
  }
  class AppScope {
    +cwd: PathBuf
    +mcp_servers: Vec~McpServer~
    +skill: String
  }
  class ChatEvent {
    <<enum>>
    MessageChunk
    ToolCall
    ToolResult
    PermissionRequest
    Usage
    Done
    Error
  }

  SidebarChat --> AgentConnection : conn
  SidebarChat ..> AppScope : new_session from
  SidebarChat ..> ChatEvent : emits
```

The `SidebarChat` reuses the `AcpAgentBackend` drain loop, but forwards each `SessionNotification` as a `ChatEvent` on the Tauri `Channel` instead of accumulating a single `String`.

## 6. App-scope assembly

```mermaid
flowchart TB
  P["notebook viewState.path change"] --> Q{spur-app.json present?}
  Q -- no --> D["Notebook app scope<br/>cwd = notebook dir<br/>mcp = foundation tools<br/>skill = notebook authoring"]
  Q -- yes --> A["read spur-app.json + skill/"]
  A --> S["App scope<br/>cwd = app dir<br/>mcp = foundation + app mcp_server<br/>skill = app SKILL.md"]
  D --> R{existing session for scope?}
  S --> R
  R -- "yes & supports_load_session" --> L["load_session(id) (replay)"]
  R -- no --> N["new_session(cwd, mcp_servers) (C-refresh)"]
```

The chat header shows the active scope ("Notebook" vs the app name). Switching apps cancels any in-flight turn, then resumes or C-refreshes.

## 7. Boundaries, build items, invariants

**Net-new surface (the only real build)**

| Layer | New |
|---|---|
| Frontend | `ChatPanel.tsx`, `stores/chat.ts`, one `SIDEBAR_PANELS` entry |
| Tauri | `chat_turn` (+ `chat_sessions_list` / `chat_switch_session` / `chat_new_session` / `chat_cancel`) |
| spur-notebook | `SidebarChat` session manager (reuses `AcpAgentBackend`), app-context loader |

**Permissions = identical to the TUI**

- Interactive: `permission_tx` -> `ChatEvent::PermissionRequest` -> inline grant/deny (mirrors `App.pending_permission` + `SessionDetailView::resolve_pending_permissions`).
- Bypass: honor `AgentConfig.skip_permissions[_args|_session_mode]` via `new_session_with_bypass`.

**Invariants inherited**

- Broadcast sizing 4096 (`NativeAcpConnection`).
- Orphan teardown: `.spur/pgids/` + `Drop` killpg.
- ACP session model: `new_session` binds MCP at creation; `load_session` replays via broadcast (subscribe-before-load); `list_sessions` merged by id + cwd.
- Reactive cascade: tools paint via `notebook_push_source`; engine schedules; bus notify-only.

**Plan preconditions**

- Task 0 — API-drift check against HEAD (session methods, `AcpAgentBackend`, `permission_tx`, `notebook_push_source`, sidebar registry).

**Deferred:** Tier-2 brain-session attach; multi-window session sharing; persisted-transcript UI beyond ACP `load_session` replay.